# 02: Transfer Learning for Multilingual Natural Language Inference

## Introduction

In this notebook, we formulate Natural Language Inference (NLI) as a three-class text-pair classification problem and solve it using transfer learning with a pretrained Transformer encoder.

Rather than learning language representations entirely from scratch, we will start with a pretrained multilingual BERT model and fine-tune it on the labelled training data from the *Contradictory, My Dear Watson* competition.

Given a premise \(P\) and a hypothesis \(H\), the model predicts one of three relationships:

- **Entailment (0):** The premise supports the hypothesis.
- **Neutral (1):** The premise neither establishes nor contradicts the hypothesis.
- **Contradiction (2):** The premise conflicts with the hypothesis.

We will use the pretrained model `bert-base-multilingual-cased`. Unlike an English-only BERT model, multilingual BERT was pretrained on text from many languages, making it a more suitable starting point for this multilingual dataset.

## Objectives

By the end of this notebook, we aim to:

1. Understand the transfer-learning approach to NLP classification.
2. Load a pretrained multilingual BERT tokenizer and encoder.
3. Encode premise–hypothesis pairs as Transformer inputs.
4. Construct a three-class sequence-pair classifier.
5. Split the labelled data into training and validation sets.
6. Fine-tune the pretrained model using PyTorch.
7. Evaluate the model using accuracy, macro-F1, and a confusion matrix.
8. Examine misclassified examples and understand the model's limitations.
9. Generate predictions in the format required for a Kaggle submission.

## Methodology

We will follow this sequence:

1. Project setup and imports
2. Load and prepare the data
3. Create a stratified training–validation split
4. Load the pretrained tokenizer and model
5. Tokenize premise–hypothesis pairs
6. Construct PyTorch datasets and data loaders
7. Fine-tune the model
8. Evaluate and analyze predictions
9. Generate the Kaggle submission file

## Experimental principles

- Keep the official competition test set separate from training and validation.
- Use a fixed random seed for reproducibility.
- Use the same validation split when comparing experiments.
- Track accuracy and macro-F1.
- Record the model checkpoint, hyperparameters, and validation results.

This notebook establishes our first pretrained Transformer baseline. More advanced models and experiments can be introduced after we understand this pipeline.

## 1. Project Setup and Imports

We begin by importing the libraries required for data manipulation, reproducible splitting, Transformer-based classification, and evaluation.

We will use:

- **pandas and NumPy** for data handling.
- **scikit-learn** for stratified splitting and evaluation metrics.
- **PyTorch** for tensor operations and training.
- **Hugging Face Transformers** for the pretrained multilingual BERT tokenizer and sequence-classification model.
- **Matplotlib** and **Seaborn** for visualizing evaluation results.

We will also define the data paths, random seed, label mapping, and model configuration in one place so that the experiment is easier to reproduce.

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Project paths
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

# Labels
LABEL_NAMES = {
    0: "Entailment",
    1: "Neutral",
    2: "Contradiction",
}

NUM_LABELS = len(LABEL_NAMES)

# Pretrained multilingual encoder
MODEL_NAME = "bert-base-multilingual-cased"

# Initial experimental configuration
VALIDATION_SIZE = 0.2
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())
print("Model:", MODEL_NAME)
print("Number of classes:", NUM_LABELS)
print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)

PyTorch version: 2.13.0
CUDA available: False
MPS available: True
Model: bert-base-multilingual-cased
Number of classes: 3
Project directory: /Users/amitkumar/Desktop/Computing/Projects/contradictory-my-dear-watson
Data directory: /Users/amitkumar/Desktop/Computing/Projects/contradictory-my-dear-watson/data


In [2]:
# verify the setup 
print("Training file exists:", TRAIN_PATH.exists())
print("Test file exists:", TEST_PATH.exists())
print("Sample submission exists:", SAMPLE_SUBMISSION_PATH.exists())

assert TRAIN_PATH.exists(), f"Missing file: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Missing file: {TEST_PATH}"
assert SAMPLE_SUBMISSION_PATH.exists(), (
    f"Missing file: {SAMPLE_SUBMISSION_PATH}"
)

Training file exists: True
Test file exists: True
Sample submission exists: True


## 2. Load and Inspect the Data

We load the labelled training set, the unlabelled competition test set, and the sample submission file.

The training set will be used to construct our training and validation subsets. The official test set will remain untouched until we are ready to generate competition predictions.

We first inspect the dimensions, column names, and representative rows to confirm that the files match our expectations.

### 2.1: Load and Read CSV

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

print("\nTraining columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("Submission columns:", sample_submission.columns.tolist())

Training shape: (12120, 6)
Test shape: (5195, 5)
Sample submission shape: (5195, 2)

Training columns: ['id', 'premise', 'hypothesis', 'lang_abv', 'language', 'label']
Test columns: ['id', 'premise', 'hypothesis', 'lang_abv', 'language']
Submission columns: ['id', 'prediction']


### 2.2: Inspect Representative Examples

In [4]:
pd.set_option("display.max_colwidth", None)

display(
    train_df[
        ["premise", "hypothesis", "lang_abv", "label"]
    ].head(5)
)

,premise,hypothesis,lang_abv,label
0,and these comments were considered in formulating the interim rules.,The rules developed in the interim were put together with these comments in mind.,en,0
1,"These are issues that we wrestle with in practice groups of law firms, she said.",Practice groups are not permitted to work on these issues.,en,2
2,Des petites choses comme celles-là font une différence énorme dans ce que j'essaye de faire.,J'essayais d'accomplir quelque chose.,fr,0
3,you know they can't really defend themselves like somebody grown uh say my age you know yeah,They can't defend themselves because of their age.,en,0
4,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสดงออกและได้เล่นหลายบทบาทไปพร้อมกัน ๆ อาจช่วยให้เด็กจับความคล้ายคลึงและความแตกต่างระหว่างผู้คนในด้านความปรารถนา ความเชื่อ และความรู้สึกได้,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,1


Let's check the test set and submission format:

In [5]:
display(test_df.head(3))
display(sample_submission.head(3))

,id,premise,hypothesis,lang_abv,language
0,c6d58c3f69,بکس، کیسی، راہیل، یسعیاہ، کیلی، کیلی، اور کولمبین ہائی اسکول کے دوسرے طلبا کے نام سے بکسوں کو نشان زد کیا جائے گا جس نے اس سال پہلے اپنی زندگی کھو دی,"کیسی کے لئے کوئی یادگار نہیں ہوگا, کولمین ہائی اسکول کے طالب علموں میں سے ایک جو مر گیا.",ur,Urdu
1,cefcc82292,هذا هو ما تم نصحنا به.,عندما يتم إخبارهم بما يجب عليهم فعله ، فشلت الإدارة في السماح لنا بالدخول إلى الأسرار التجارية.,ar,Arabic
2,e98005252c,et cela est en grande partie dû au fait que les mères prennent de la drogue,Les mères se droguent.,fr,French


,id,prediction
0,c6d58c3f69,1
1,cefcc82292,1
2,e98005252c,1


### 2.3: Validate Labels

In [6]:
print("Missing values in training data:")
display(train_df.isna().sum())

print("\nTraining label counts:")
display(train_df["label"].value_counts().sort_index())

print("\nUnique labels:", sorted(train_df["label"].unique()))

assert set(train_df["label"].unique()) == {0, 1, 2}
assert train_df["label"].notna().all()

Missing values in training data:


id            0
premise       0
hypothesis    0
lang_abv      0
language      0
label         0
dtype: int64


Training label counts:


label
0    4176
1    3880
2    4064
Name: count, dtype: int64


Unique labels: [np.int64(0), np.int64(1), np.int64(2)]


## 3. Stratified Training–Validation Split

Before fine-tuning the model, we divide the labelled training data into two disjoint subsets:

- **Training set:** Used to update the model's parameters.
- **Validation set:** Used to evaluate the model on examples that were not used for parameter updates.

We reserve 20% of the labelled data for validation and use the remaining 80% for training.

### Why stratification?

The target has three classes: entailment, neutral, and contradiction. A stratified split approximately preserves their proportions in both subsets.

This makes the comparison between training and validation results more interpretable, particularly when the class frequencies are not exactly equal.

We fix the random seed to make the split reproducible. The official Kaggle test set remains separate and will not be used for training or validation.

In [7]:
train_split, val_split = train_test_split(
    train_df,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=train_df["label"],
)

train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

print("Training subset:", train_split.shape)
print("Validation subset:", val_split.shape)

Training subset: (9696, 6)
Validation subset: (2424, 6)


In [8]:
# verify the class proportions
split_distribution = pd.DataFrame({
    "Full training set": train_df["label"].value_counts(normalize=True),
    "Training subset": train_split["label"].value_counts(normalize=True),
    "Validation subset": val_split["label"].value_counts(normalize=True),
}).sort_index()

split_distribution.index = split_distribution.index.map(LABEL_NAMES)

display(split_distribution)

,Full training set,Training subset,Validation subset
label,,,
Entailment,0.344554,0.344575,0.344472
Neutral,0.320132,0.320132,0.320132
Contradiction,0.335314,0.335293,0.335396


## 4. Loading the Pretrained Multilingual BERT Tokenizer and Model

We use `bert-base-multilingual-cased`, a pretrained Transformer encoder designed to process text in multiple languages.

There are two components to load:

1. **Tokenizer:** Converts raw text into token IDs and constructs the attention masks required by BERT.
2. **Sequence-classification model:** Contains the pretrained BERT encoder and a classification head for predicting the three NLI labels.

The pretrained encoder provides representations learned from large-scale multilingual text. The classification head is initialized for our downstream task and will be adapted during fine-tuning.

We will use the same pretrained model identifier for both components to ensure that the tokenizer and encoder are compatible.

### 4.1: Select Device

In [9]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)

Using device: mps


### 4.2: Load Tokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum model input length:", tokenizer.model_max_length)

Tokenizer: bert-base-multilingual-cased
Vocabulary size: 119547
Maximum model input length: 512


### 4.3: Load The Classification Model

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

model = model.to(DEVICE)

print("Model loaded successfully.")
print("Number of labels:", model.config.num_labels)
print("Device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully.
Number of labels: 3
Device: mps:0


In [12]:
# verify 
print(model.classifier)

print(
    "Output dimension:",
    model.classifier.out_features,
)

assert model.config.num_labels == 3
assert model.classifier.out_features == 3

Linear(in_features=768, out_features=3, bias=True)
Output dimension: 3


## 5. Tokenizing Premise–Hypothesis Pairs

BERT does not directly process raw text. It receives numerical representations of text produced by a tokenizer.

For Natural Language Inference, we supply the premise and hypothesis together as a **sentence pair**. The tokenizer converts them into token IDs and constructs an attention mask.

We will study four components:

1. **Special tokens:** Mark the beginning of the input and the boundary between sentences.
2. **Input IDs:** Integer indices representing tokens in the model's vocabulary.
3. **Attention mask:** Indicates which positions contain actual tokens rather than padding.
4. **Truncation and padding:** Ensure sequences fit within a chosen maximum length and can be batched together.

We will first inspect one example before applying tokenization to the entire dataset.

In [13]:
# tokenize a sample example 
premise = "A person is riding a bicycle."
hypothesis = "Someone is cycling."

example_encoding = tokenizer(
    premise,
    hypothesis,
    truncation=True,
    max_length=MAX_LENGTH,
)

print(example_encoding)

{'input_ids': [101, 138, 15042, 10124, 56629, 169, 11342, 49274, 119, 102, 94524, 10124, 171, 87497, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [14]:
# inspect the tokens
input_ids = example_encoding["input_ids"]

tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Tokens:")
print(tokens)

print("\nInput IDs:")
print(input_ids)

print("\nAttention mask:")
print(example_encoding["attention_mask"])

print("\nDecoded sequence:")
print(tokenizer.decode(input_ids))

Tokens:
['[CLS]', 'A', 'person', 'is', 'riding', 'a', 'bi', '##cycle', '.', '[SEP]', 'Someone', 'is', 'c', '##ycling', '.', '[SEP]']

Input IDs:
[101, 138, 15042, 10124, 56629, 169, 11342, 49274, 119, 102, 94524, 10124, 171, 87497, 119, 102]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Decoded sequence:
[CLS] A person is riding a bicycle. [SEP] Someone is cycling. [SEP]


### Interpreting the encoded pair

For BERT's paired-sentence input, the sequence follows this structure:

$$
[\mathrm{CLS}]\; P\; [\mathrm{SEP}]\; H\; [\mathrm{SEP}]
$$

Here:

- `[CLS]` marks the beginning of the sequence and provides a representation used by the classification head.
- The first `[SEP]` marks the end of the premise.
- The second `[SEP]` marks the end of the hypothesis.

The tokenizer also returns `token_type_ids`, which distinguish tokens belonging to the first and second sequences in the standard BERT input format.

The attention mask distinguishes actual tokens from padding tokens. In this example, because we have not requested padding, the mask should consist entirely of ones.

In [15]:
# examine token type ids
if "token_type_ids" in example_encoding:
    print("Token type IDs:")
    print(example_encoding["token_type_ids"])

    print("\nTokens and their sequence IDs:")
    for token, token_type_id in zip(
        tokens,
        example_encoding["token_type_ids"],
    ):
        print(f"{token:15s} sequence={token_type_id}")
else:
    print("This tokenizer does not return token_type_ids.")

Token type IDs:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]

Tokens and their sequence IDs:
[CLS]           sequence=0
A               sequence=0
person          sequence=0
is              sequence=0
riding          sequence=0
a               sequence=0
bi              sequence=0
##cycle         sequence=0
.               sequence=0
[SEP]           sequence=0
Someone         sequence=1
is              sequence=1
c               sequence=1
##ycling        sequence=1
.               sequence=1
[SEP]           sequence=1


For standard BERT pair encoding, the premise and its initial special token have sequence ID 0; the hypothesis and its separator have sequence ID 1, while [CLS] belongs to sequence 0.

In [16]:
# observe padding 
padded_encoding = tokenizer(
    premise,
    hypothesis,
    padding="max_length",
    truncation=True,
    max_length=32,
)

print("Input length:", len(padded_encoding["input_ids"]))
print("Tokens:")
print(tokenizer.convert_ids_to_tokens(padded_encoding["input_ids"]))

print("\nAttention mask:")
print(padded_encoding["attention_mask"])

Input length: 32
Tokens:
['[CLS]', 'A', 'person', 'is', 'riding', 'a', 'bi', '##cycle', '.', '[SEP]', 'Someone', 'is', 'c', '##ycling', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 6. Tokenizing the Datasets

We now apply the pretrained tokenizer to the training, validation, and competition test sets.

For each example, the tokenizer receives the premise and hypothesis as a pair. We enable truncation with a maximum sequence length of 128 tokens as an initial experimental choice.

We do not pad each example to the maximum length at this stage. Instead, we will use dynamic padding when constructing batches, reducing unnecessary computation on padding tokens.

The labels are kept separately from the tokenized inputs. Training and validation examples have labels, whereas the official test examples do not.

In [17]:
# define tokenization function 
def tokenize_pairs(dataframe):
    return tokenizer(
        dataframe["premise"].tolist(),
        dataframe["hypothesis"].tolist(),
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

In [18]:
# tokenize each subsets 
train_encodings = tokenize_pairs(train_split)
val_encodings = tokenize_pairs(val_split)
test_encodings = tokenize_pairs(test_df)

print("Training examples:", len(train_encodings["input_ids"]))
print("Validation examples:", len(val_encodings["input_ids"]))
print("Test examples:", len(test_encodings["input_ids"]))

Training examples: 9696
Validation examples: 2424
Test examples: 5195


In [19]:
# inspect tokenization lengths
train_token_lengths = [
    len(ids) for ids in train_encodings["input_ids"]
]

print("Minimum token length:", min(train_token_lengths))
print("Median token length:", np.median(train_token_lengths))
print("95th percentile:", np.percentile(train_token_lengths, 95))
print("Maximum token length:", max(train_token_lengths))

print(
    "Training examples reaching the maximum length:",
    sum(length == MAX_LENGTH for length in train_token_lengths),
)

Minimum token length: 8
Median token length: 45.0
95th percentile: 91.0
Maximum token length: 128
Training examples reaching the maximum length: 91


## 7. PyTorch Dataset and DataLoader

The tokenizer produces lists of token IDs and attention masks, but a training loop needs batches of tensors.

We will implement a custom PyTorch `Dataset` that returns the tokenized inputs for one example and its label when available.

A `DataLoader` will collect examples into batches. Because examples have different sequence lengths, we use `DataCollatorWithPadding` to pad each batch to its longest sequence rather than padding every example to the same fixed length.

This reduces unnecessary padding and can improve computational efficiency.

The official test dataset has no labels, so its dataset will return only model inputs.

In [20]:
# define the dataset 
class NLIDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {
            key: values[idx]
            for key, values in self.encodings.items()
        }

        if self.labels is not None:
            item["labels"] = int(self.labels[idx])

        return item

In [21]:
# construct datasets and collator
train_dataset = NLIDataset(
    train_encodings,
    train_split["label"].tolist(),
)

val_dataset = NLIDataset(
    val_encodings,
    val_split["label"].tolist(),
)

test_dataset = NLIDataset(test_encodings)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

In [22]:
# construct dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 606
Validation batches: 152
Test batches: 325


In [23]:
# inspect one batch
batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")

print("\nLabels in this batch:")
print(batch["labels"])

print("\nAttention mask for the batch:")
print(batch["attention_mask"])

input_ids: shape=(16, 79), dtype=torch.int64
token_type_ids: shape=(16, 79), dtype=torch.int64
attention_mask: shape=(16, 79), dtype=torch.int64
labels: shape=(16,), dtype=torch.int64

Labels in this batch:
tensor([1, 2, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 1, 1, 2, 0])

Attention mask for the batch:
tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])


**Sanity Check:** Let's inspect language representation in our split:

In [24]:
language_distribution = pd.DataFrame({
    "Full dataset": train_df["language"].value_counts(normalize=True),
    "Training split": train_split["language"].value_counts(normalize=True),
    "Validation split": val_split["language"].value_counts(normalize=True),
}).fillna(0)

display(
    (100 * language_distribution).round(2).sort_values(
        "Full dataset", ascending=False
    )
)

,Full dataset,Training split,Validation split
language,,,
English,56.68,56.89,55.86
Chinese,3.39,3.18,4.25
Arabic,3.31,3.32,3.26
French,3.22,3.41,2.43
Swahili,3.18,3.23,2.97
Urdu,3.14,3.21,2.89
Vietnamese,3.13,3.22,2.76
Russian,3.10,2.91,3.88
Hindi,3.09,2.97,3.55


## 8. Training the Model

We fine-tune multilingual BERT on the labelled NLI training subset. Fine-tuning updates the pretrained encoder parameters as well as the newly initialized classification head.

### Loss function

We use cross-entropy loss for the three-class classification task. Given logits \(z\) and the true class \(y\), the loss is

$$
\mathcal{L}=-\log\left(
\frac{e^{z_y}}{\sum_{j=0}^{2}e^{z_j}}
\right).
$$

The model produces three logits per example. Cross-entropy compares these logits with the true class labels.

### Optimizer

We use AdamW with an initial learning rate of $2\times10^{-5}$. This is a common starting point for Transformer fine-tuning.

### Training procedure

We train for three epochs and evaluate on the validation set after each epoch. We track accuracy and macro-F1, saving the model with the best validation macro-F1.

Macro-F1 calculates F1 independently for each class and then averages the three scores equally. This provides a useful complement to accuracy when evaluating classification performance.

The official competition test set is not used for checkpoint selection.

In [25]:
# Configure the optimizer and loss
import copy
import time

import torch.nn as nn
from torch.optim import AdamW

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", LEARNING_RATE)
print("Loss function:", type(criterion).__name__)
print("Epochs:", NUM_EPOCHS)
print("Device:", DEVICE)

Optimizer: AdamW
Learning rate: 2e-05
Loss function: CrossEntropyLoss
Epochs: 3
Device: mps


In [31]:
# define the evaluation function 
def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_predictions = []
    all_labels = []

    progress_bar = tqdm(
        data_loader,
        desc="Validation",
        leave=True,
    )

    with torch.no_grad():
        for batch in progress_bar:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            labels = batch["labels"]

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                token_type_ids=batch.get("token_type_ids"),
            )

            loss = criterion(outputs.logits, labels)
            total_loss += loss.item() * labels.size(0)

            predictions = outputs.logits.argmax(dim=1)
            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            progress_bar.set_postfix(
                loss=f"{total_loss / len(all_labels):.4f}"
            )

    average_loss = total_loss / len(data_loader.dataset)
    accuracy = accuracy_score(all_labels, all_predictions)
    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "predictions": all_predictions,
        "labels": all_labels,
    }

In [32]:
# to display progress bar 
from tqdm.auto import tqdm

In [33]:
# define the training function 
def train_one_epoch(model, data_loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_examples = 0

    progress_bar = tqdm(
        data_loader,
        desc="Training",
        leave=True,
    )

    for batch in progress_bar:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        labels = batch["labels"]

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch.get("token_type_ids"),
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        progress_bar.set_postfix(
            loss=f"{total_loss / total_examples:.4f}"
        )

    return total_loss / total_examples

In [34]:
# train and select the best checkpoint 
history = []
best_macro_f1 = -1.0
best_epoch = None
best_model_state = None

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE,
    )

    val_metrics = evaluate_model(
        model,
        val_loader,
        criterion,
        DEVICE,
    )

    elapsed = time.time() - start_time

    epoch_record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "seconds": elapsed,
    }

    history.append(epoch_record)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_accuracy={val_metrics['accuracy']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
        f"time={elapsed:.1f}s"
    )

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch + 1
        best_model_state = copy.deepcopy(model.state_dict())

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation macro-F1: {best_macro_f1:.4f}")

Training:   0%|          | 0/606 [00:00<?, ?it/s]

Validation:   0%|          | 0/152 [00:00<?, ?it/s]

Epoch 1/3 | train_loss=0.9090 | val_loss=0.8325 | val_accuracy=0.6250 | val_macro_f1=0.6231 | time=33451.3s


Training:   0%|          | 0/606 [00:00<?, ?it/s]

Validation:   0%|          | 0/152 [00:00<?, ?it/s]

Epoch 2/3 | train_loss=0.6633 | val_loss=0.7839 | val_accuracy=0.6741 | val_macro_f1=0.6741 | time=13358.8s


Training:   0%|          | 0/606 [00:00<?, ?it/s]

Validation:   0%|          | 0/152 [00:00<?, ?it/s]

Epoch 3/3 | train_loss=0.4237 | val_loss=0.9800 | val_accuracy=0.6510 | val_macro_f1=0.6508 | time=34668.8s

Best epoch: 2
Best validation macro-F1: 0.6741


**Save the best checkpoint**

In [35]:
from pathlib import Path

checkpoint_dir = Path("../checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "mbert_nli_best.pt"

torch.save(
    {
        "epoch": best_epoch,
        "model_state_dict": best_model_state,
        "best_val_macro_f1": best_macro_f1,
        "model_name": MODEL_NAME,
    },
    checkpoint_path,
)

print("Saved checkpoint to:", checkpoint_path)

Saved checkpoint to: ../checkpoints/mbert_nli_best.pt


## 9. Per-language Evaluation

### 9.1: Restore the best model

In [36]:
# Restore the checkpoint with the best validation macro-F1
model.load_state_dict(best_model_state)
model.eval()

print(f"Evaluating best checkpoint from epoch {best_epoch}")

Evaluating best checkpoint from epoch 2


### 9.2: Generate validation predictions

In [37]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validation predictions"):
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        labels = batch.pop("labels")

        outputs = model(**batch)
        predictions = outputs.logits.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_predictions)

print("Validation examples:", len(y_true))

Validation predictions:   0%|          | 0/152 [00:00<?, ?it/s]

Validation examples: 2424


### 9.3: Overall classification report

In [38]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

label_names = ["entailment", "neutral", "contradiction"]

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=label_names,
        digits=4,
        zero_division=0
    )
)

display(
    pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
        index=[f"Actual {name}" for name in label_names],
        columns=[f"Predicted {name}" for name in label_names]
    )
)

               precision    recall  f1-score   support

   entailment     0.7050    0.6467    0.6746       835
      neutral     0.6617    0.6856    0.6734       776
contradiction     0.6581    0.6913    0.6743       813

     accuracy                         0.6741      2424
    macro avg     0.6749    0.6745    0.6741      2424
 weighted avg     0.6754    0.6741    0.6741      2424



,Predicted entailment,Predicted neutral,Predicted contradiction
Actual entailment,540,140,155
Actual neutral,107,532,137
Actual contradiction,119,132,562


### 9.4 Per-language classification report

In [39]:
from sklearn.metrics import accuracy_score, f1_score

val_languages = val_split["language"].reset_index(drop=True)

assert len(val_languages) == len(y_true)

language_results = []

for language in sorted(val_languages.unique()):
    mask = (val_languages == language).to_numpy()

    language_results.append({
        "language": language,
        "examples": int(mask.sum()),
        "accuracy": accuracy_score(y_true[mask], y_pred[mask]),
        "macro_f1": f1_score(
            y_true[mask],
            y_pred[mask],
            labels=[0, 1, 2],
            average="macro",
            zero_division=0
        )
    })

language_report = (
    pd.DataFrame(language_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(
    language_report.style.format({
        "accuracy": "{:.4f}",
        "macro_f1": "{:.4f}"
    })
)

,language,examples,accuracy,macro_f1
0,French,59,0.7288,0.7291
1,Chinese,103,0.7184,0.7201
2,Vietnamese,67,0.7164,0.7186
3,German,67,0.7164,0.7134
4,English,1354,0.7001,0.7003
5,Spanish,83,0.6988,0.6995
6,Arabic,79,0.6962,0.6982
7,Hindi,86,0.6860,0.6743
8,Bulgarian,71,0.6197,0.6221
9,Russian,94,0.6277,0.6142


**Observation:** English constitutes approximately 56% of the dataset, but its validation performance is comparable to that of several other languages. The current results do not establish that English data volume or sentence length dominates model behavior. Further analysis would be needed to investigate these effects.

## 10. Generate Predictions on the Official Test Set

### 10.1: Restore the best checkpoint

In [40]:
# Restore the best model checkpoint
model.load_state_dict(best_model_state)

# Set the model to evaluation mode
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,)

### 10.2: Generate test predictions

In [41]:
# Generate predictions for the official test set

test_predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test predictions"):

        # Move inputs to the device
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        token_type_ids = batch["token_type_ids"].to(DEVICE)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # Predicted class for each example
        predictions = torch.argmax(outputs.logits, dim=1)

        test_predictions.extend(
            predictions.cpu().numpy()
        )

print("Number of test predictions:", len(test_predictions))

Test predictions:   0%|          | 0/325 [00:00<?, ?it/s]

Number of test predictions: 5195


### 10.3: Create the Kaggle submission file

In [42]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": test_predictions
})

display(submission.head())

,id,prediction
0,c6d58c3f69,2
1,cefcc82292,1
2,e98005252c,0
3,58518c10ba,1
4,c32b0d16df,2


In [43]:
# Check that the predictions and test examples match
assert len(submission) == len(test_df)
assert submission["id"].equals(test_df["id"])

print("Submission shape:", submission.shape)
print("Prediction distribution:")
display(submission["prediction"].value_counts().sort_index())

Submission shape: (5195, 2)
Prediction distribution:


prediction
0    1686
1    1750
2    1759
Name: count, dtype: int64

In [44]:
# save the file 
submission.to_csv("submission.csv", index=False)

**Verify the submission file**

In [45]:
display(submission.head())
print(submission.columns.tolist())

,id,prediction
0,c6d58c3f69,2
1,cefcc82292,1
2,e98005252c,0
3,58518c10ba,1
4,c32b0d16df,2


['id', 'prediction']


In [48]:
from pathlib import Path

# Find the data folder relative to the notebook's current directory
data_dir = Path("../data")

print("Files in data folder:")
print([p.name for p in data_dir.iterdir()])

Files in data folder:
['.DS_Store', 'test.csv', 'train.csv', 'sample_submission.csv']


In [49]:
# load ssample_submission
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

display(sample_submission.head())
print("Expected columns:", sample_submission.columns.tolist())
print("Expected shape:", sample_submission.shape)

,id,prediction
0,c6d58c3f69,1
1,cefcc82292,1
2,e98005252c,1
3,58518c10ba,1
4,c32b0d16df,1


Expected columns: ['id', 'prediction']
Expected shape: (5195, 2)


In [50]:
# Verify submission format before uploading

assert submission.columns.tolist() == sample_submission.columns.tolist()
assert len(submission) == len(sample_submission)
assert submission["id"].equals(sample_submission["id"])

print("Submission format verified!")
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())

Submission format verified!
Rows: 5195
Columns: ['id', 'prediction']


In [51]:
# save the file to project's data directory
submission.to_csv("../data/submission.csv", index=False)

In [58]:
from pathlib import Path

print("Model-related files:")
for path in Path("../").rglob("*"):
    if path.is_file() and (
        "model" in path.name.lower()
        or "checkpoint" in path.name.lower()
    ):
        print(path)

Model-related files:
../notebooks/.ipynb_checkpoints/02_classical_nli_baseline-checkpoint.ipynb
../notebooks/.ipynb_checkpoints/01_nli_fundamentals-checkpoint.ipynb


## 11. Conclusion

We established a multilingual BERT baseline for three-class natural language inference on the Contradictory, My Dear Watson competition dataset.

The model was fine-tuned using cross-entropy loss and AdamW, with validation macro-F1 used to select the best checkpoint. Training was conducted on Kaggle to take advantage of its computational resources.

The best checkpoint was selected from epoch 2, achieving a validation macro-F1 of 0.6741. We subsequently generated predictions for the official competition test set and submitted them to Kaggle.

### Results

- Best validation macro-F1: 0.6741
- Best epoch: 2
- Kaggle submission score: 0.66025
- Submission status: Submitted

### Limitations and future work

The submission provides an initial benchmark for subsequent experiments. Although validation performance varied across languages, we have not established that the English-language examples dominated the model's performance.

Further work will investigate improvements in multilingual generalization, alternative pretrained models, and training configurations. These experiments will be documented in separate notebooks to preserve the reproducibility and clarity of this baseline.
